In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,hidden_dim,head_num,attention_dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_num = head_num
        assert hidden_dim % head_num == 0
        self.head_dim = hidden_dim // head_num
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim)

        self.attention_dropout = nn.Dropout(attention_dropout)
    
    def forward(self,X,attention_mask=None):
        batch,seq_len,_ = X.size() #因为第三个值我们已知是head_dim*head_num
        Q = self.q_proj(X) 
        K = self.k_proj(X)
        V = self.v_proj(X)    # [batch,seq_len,head_dim*head_num]
        q_state= Q.view(batch,seq_len,self.head_num,self.head_dim).transpose(1,2)
        k_state= K.view(batch,seq_len,self.head_num,self.head_dim).transpose(1,2)
        v_state= V.view(batch,seq_len,self.head_num,self.head_dim).transpose(1,2)

        attention_weight = torch.matmul(
            q_state,k_state.transpose(-1,-2)
        )/math.sqrt(self.head_dim)

        if attention_mask is not None:
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0, float('-inf')
            )
        print(attention_weight.shape)

        attention_weight = torch.softmax(attention_weight,-1)
        attention_weight = self.attention_dropout(attention_weight)
        output_mid = torch.matmul(
            attention_weight,v_state
        )
        output_mid = output_mid.transpose(1,2).contiguous()
        output_mid = output_mid.view(batch,seq_len,-1)

        output = self.o_proj(output_mid)
        return output




In [19]:
attention_mask = (torch.tensor([[0,1],[0,0],[1,0]]).unsqueeze(1).unsqueeze(2).expand(3, 8, 2, 2))
x = torch.rand(3, 2, 128)
net = MultiHeadAttention(128,8)
net(x,attention_mask)

torch.Size([3, 8, 2, 2])


tensor([[[-7.8944e-02,  3.0840e-01,  4.1127e-01, -4.7560e-01,  4.1513e-01,
           3.5165e-01, -1.4568e-01, -3.5204e-01, -4.3832e-01, -1.9360e-01,
          -5.0918e-02, -5.9082e-02,  2.0525e-01, -2.5752e-01, -4.0362e-01,
          -3.4303e-01, -4.0622e-02,  1.2125e-01, -1.8068e-02, -2.3495e-01,
           6.9810e-02, -2.0581e-01,  3.7819e-02,  2.5776e-01,  4.0685e-01,
          -4.6155e-02,  3.9128e-01, -2.1804e-01,  7.9739e-02, -2.6827e-01,
           4.3411e-01, -1.2030e-01, -7.9499e-02,  7.9016e-02,  1.7612e-01,
           1.1213e-02,  1.4987e-01,  4.7544e-01, -5.8133e-02,  1.5705e-01,
           4.8496e-01,  2.0492e-01, -4.2848e-01,  3.7847e-01,  7.3965e-03,
           2.7706e-01,  5.6050e-02, -3.1475e-02, -6.6621e-02, -2.8872e-01,
           2.6186e-01,  1.4862e-01, -2.6494e-01, -1.6478e-02,  6.1800e-02,
          -2.8976e-02,  1.6595e-01,  1.2941e-01,  5.7899e-02,  1.2353e-01,
           2.8351e-01,  2.1335e-01, -1.0489e-02,  1.2465e-01,  9.7569e-02,
           9.2812e-02, -3

In [4]:
class MultiQueryAttention(nn.Module):
    def __init__(self,hidden_dim,head_num,attention_dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_num = head_num
        assert hidden_dim % head_num == 0
        self.head_dim = hidden_dim // head_num
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, self.head_dim)
        self.v_proj = nn.Linear(hidden_dim, self.head_dim)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim)

        self.attention_dropout = nn.Dropout(attention_dropout)
    
    def forward(self,X,attention_mask=None):
        batch,seq_len,_ = X.size() #因为第三个值我们已知是head_dim*head_num
        # Q = self.q_proj(X) 
        # K = self.k_proj(X)
        # V = self.v_proj(X)    # [batch,seq_len,head_dim*head_num]
        # q_state= Q.view(batch,seq_len,self.head_num,self.head_dim).transpose(1,2)
        # k_state= K.view(batch,seq_len,self.head_num,self.head_dim).transpose(1,2)
        # v_state= V.view(batch,seq_len,self.head_num,self.head_dim).transpose(1,2)

        Q = self.q_proj(X) 
        K = self.k_proj(X)
        V = self.v_proj(X)    # [batch,seq_len,head_dim*head_num]
        q_state= Q.view(batch,seq_len,self.head_num,self.head_dim).transpose(1,2)
        K = K.unsqueeze(1)
        k_state= K.expand(-1,self.head_num,-1,-1)
        V = V.unsqueeze(1)
        v_state= V.expand(-1,self.head_num,-1,-1)


        attention_weight = torch.matmul(
            q_state,k_state.transpose(-1,-2)
        )/math.sqrt(self.head_dim)

        if attention_mask is not None:
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0, float('-inf')
            )
        print(attention_weight.shape)

        attention_weight = torch.softmax(attention_weight,-1)
        attention_weight = self.attention_dropout(attention_weight)
        output_mid = torch.matmul(
            attention_weight,v_state
        )
        output_mid = output_mid.transpose(1,2).contiguous()
        output_mid = output_mid.view(batch,seq_len,-1)

        output = self.o_proj(output_mid)
        return output

attention_mask = (torch.tensor([[0,1],[0,0],[1,0]]).unsqueeze(1).unsqueeze(2).expand(3, 8, 2, 2))
x = torch.rand(3, 2, 128)
net = MultiQueryAttention(128,8)
net(x,attention_mask)


torch.Size([3, 8, 2, 2])


tensor([[[-0.0324, -0.0735,  0.3448, -0.2589, -0.1378,  0.1518,  0.0606,
           0.0823,  0.0851,  0.4106, -0.3688,  0.2780, -0.0062,  0.0682,
           0.0210,  0.0250,  0.1613,  0.2233,  0.0426, -0.3335,  0.0589,
          -0.0166,  0.2535, -0.1019, -0.0450,  0.0024, -0.0133, -0.3262,
           0.2560,  0.2370,  0.1344, -0.1205,  0.1588, -0.0866,  0.0227,
           0.1999, -0.1816, -0.1005, -0.0247, -0.0258, -0.0225,  0.1079,
           0.3912, -0.0214,  0.1644, -0.3238,  0.0552, -0.0829,  0.2687,
          -0.1220, -0.1370, -0.0437,  0.0132,  0.1118,  0.3334,  0.2154,
           0.2321, -0.0120,  0.0525,  0.1011,  0.0949,  0.1474, -0.1447,
           0.0236,  0.0483, -0.2248,  0.1709,  0.0045,  0.0450,  0.1833,
          -0.1343,  0.1427,  0.0750, -0.3830,  0.1185,  0.1523,  0.0740,
          -0.0813,  0.0365,  0.0192,  0.0214,  0.2091, -0.0623, -0.1483,
           0.1331, -0.0912,  0.0469,  0.1303, -0.0375, -0.2736,  0.0039,
          -0.2223, -0.1612, -0.1471, -0.3312, -0.34

In [ ]:
class GroupQueryAttention(nn.Module):
    def __init__(self,hidden_dim,head_num,kv_heads=2,attention_dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_num = head_num
        self.kv_heads = kv_heads
        assert hidden_dim % head_num == 0
        self.head_dim = hidden_dim // head_num
        self.repeat_factor =head_num // kv_heads
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, kv_heads * self.head_dim)
        self.v_proj = nn.Linear(hidden_dim, kv_heads * self.head_dim)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim)

        self.attention_dropout = nn.Dropout(attention_dropout)
    
    def forward(self,X,attention_mask=None):
        batch,seq_len,_ = X.size() #因为第三个值我们已知是head_dim*head_num
        Q = self.q_proj(X) 
        K = self.k_proj(X)
        V = self.v_proj(X)    # [batch,seq_len,head_dim*head_num]
        q_state= Q.view(batch,seq_len,self.head_num,self.head_dim).transpose(1,2)
        k_state= K.view(batch,seq_len,self.kv_heads,self.head_dim).transpose(1,2)
        v_state= V.view(batch,seq_len,self.kv_heads,self.head_dim).transpose(1,2)
        k_state = k_state.repeat_interleave(self.repeat_factor, dim=1)
        v_state = v_state.repeat_interleave(self.repeat_factor, dim=1)


        attention_weight = torch.matmul(
            q_state,k_state.transpose(-1,-2)
        )/math.sqrt(self.head_dim)

        if attention_mask is not None:
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0, float('-inf')
            )
        print(attention_weight.shape)

        attention_weight = torch.softmax(attention_weight,-1)
        attention_weight = self.attention_dropout(attention_weight)
        output_mid = torch.matmul(
            attention_weight,v_state
        )
        output_mid = output_mid.transpose(1,2).contiguous()
        output_mid = output_mid.view(batch,seq_len,-1)

        output = self.o_proj(output_mid)
        return output

attention_mask = (torch.tensor([[0,1],[0,0],[1,0]]).unsqueeze(1).unsqueeze(2).expand(3, 8, 2, 2))
x = torch.rand(3, 2, 128)
net = GroupQueryAttention(128,8)
net(x,attention_mask)


torch.Size([3, 8, 2, 2])


tensor([[[ 1.1813e-01,  3.1654e-02, -2.2118e-01, -1.6738e-01, -2.4348e-01,
          -1.2024e-01, -5.4889e-02, -9.8047e-02, -4.2029e-01, -3.3550e-02,
           3.0732e-01, -5.3061e-02, -2.1977e-01, -6.7312e-02, -1.9905e-02,
           7.9828e-02, -9.2899e-02,  7.7376e-02,  2.4179e-01,  2.0056e-01,
           2.2379e-01, -2.6609e-01, -1.8624e-01,  1.0005e-01,  3.3743e-01,
          -2.0621e-01,  4.0137e-02, -1.1141e-01, -4.0585e-03,  1.0451e-01,
           1.3092e-01,  1.5189e-01,  9.1824e-02, -9.0585e-02, -2.2001e-02,
           1.5671e-01,  1.4616e-01,  3.4993e-01, -7.2038e-02,  4.7488e-01,
          -1.0413e-01,  4.6320e-01,  1.9618e-01, -3.2439e-01, -4.8554e-02,
          -1.8227e-01, -3.1288e-01,  4.3123e-01,  1.1523e-01,  1.0516e-01,
          -2.2267e-01, -1.7750e-01,  6.2420e-02,  2.0755e-01,  3.6960e-01,
           2.4275e-01,  1.4594e-01, -7.7811e-02, -2.1109e-01, -9.0228e-02,
          -1.4527e-01, -9.4040e-02, -1.5575e-01, -2.7690e-01, -1.3621e-01,
           7.5627e-02,  2